In [ ]:
from google.cloud import bigquery
import pandas as pd
import pandas
from decimal import Decimal
import queue, concurrent.futures
import traceback
import time
import threading
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
from matplotlib_venn import venn2, venn3

from pandas.tseries.offsets import CustomBusinessDay



In [ ]:
PROJECT_ID = "clean-pilot-456915-t0"
client = bigquery.Client(project= PROJECT_ID)
df = client.query("""  
SELECT 
    t.id,
    t.tranid,
    t.trandate,
    t.status,
    t.type,
    tl.item,
    i.fullname,
    i.class,
    i.manufacturer,          
    c.category,             
    tl.quantity,
    tl.quantitybackordered,
    tl.netamount,
    tl.rate,
    tl.costestimate,
    tl.inventorylocation,    
    tl.custcolfree_goods_checkbox  
FROM `clean-pilot-456915-t0.NetSuite.transaction` t
JOIN `clean-pilot-456915-t0.NetSuite.transactionline` tl ON t.id = tl.transaction
JOIN `clean-pilot-456915-t0.NetSuite.customer` c ON t.entity = c.id
LEFT JOIN `clean-pilot-456915-t0.NetSuite.item` i ON tl.item = i.id
WHERE t.trandate <= DATE_SUB(CURRENT_DATE(), INTERVAL 1 DAY)
  AND t.type IN ('CustInvc')   
  AND t.entity NOT IN (604610, 615265)
  AND c.category != 6                      -- exclude wholesale
  AND tl.mainline = FALSE
  AND tl.taxline = FALSE
  AND tl.inventorylocation = 1
  AND tl.itemtype = 'InvtPart'
  AND i.manufacturer != 'D2'
ORDER BY t.trandate DESC
""").to_dataframe()

In [ ]:
inv = df.groupby(by = ["tranid", "item"], as_index=False).agg(
    quantity = ("quantity", "sum"), netamount = ("netamount", "sum"),
    costestimate = ("costestimate", "sum"), status = ("status","first"),
    trandate = ("trandate", "first"), type = ("type", "first")
)
# Selected the Analysis timeline
six_months_ago = (datetime.today() - timedelta(days=182)).date()
inv = inv[inv["trandate"] >= six_months_ago]

# Adjust the quantitative columns
adj_list = ["quantity", "netamount","costestimate"]
for col in adj_list:
    inv[col] = inv[col]*-1

# Include 'Paid In Full' only - B
valid_inv = inv[inv["status"] == 'B']

valid_inv["margin"] = valid_inv["netamount"] - valid_inv["costestimate"]
item_inv_agg = valid_inv.groupby(by = "item", as_index=False).agg(
    total_amount = ("netamount", "sum"),
    total_quantity = ("quantity", "sum"),
    total_costestimate = ("costestimate", "sum"),
    total_margin = ("margin", "sum")
)

In [ ]:
# Top revenue - amount
item_inv_agg_amount = (item_inv_agg.sort_values(by= "total_amount", ascending=False)
                .assign(cum_pct = lambda x: x["total_amount"].cumsum()/ x["total_amount"].sum())
                .reset_index(drop = True))
threshold = 0.8
x_val = item_inv_agg_amount["cum_pct"].ge(threshold).idxmax()

top_revenue = item_inv_agg_amount.iloc[0:x_val, :]

# Top margin - margin 
item_inv_agg_margin = (item_inv_agg.sort_values(by= "total_margin", ascending=False)
                .assign(cum_pct = lambda x: x["total_margin"].cumsum()/ x["total_margin"].sum())
                .reset_index(drop = True))
threshold = 0.8
x_val = item_inv_agg_margin["cum_pct"].ge(threshold).idxmax()
top_margin = item_inv_agg_margin.iloc[0:x_val, :]

# Top quantity - quantity
item_inv_agg_quantity = (item_inv_agg.sort_values(by= "total_quantity", ascending=False)
                .assign(cum_pct = lambda x: x["total_quantity"].cumsum()/ x["total_quantity"].sum())
                .reset_index(drop = True))
threshold = 0.8
x_val = item_inv_agg_quantity["cum_pct"].ge(threshold).idxmax()
top_quantity = item_inv_agg_quantity.iloc[0:x_val, :]

In [ ]:
item_col = "item"
set1 = set(top_revenue[item_col]) 
set2 = set(top_margin[item_col])
set3 = set(top_quantity[item_col])

fig, axes = plt.subplots(2, 2, figsize=(10,8))

venn2([set1,set2], set_labels=("Top Revenue", "Top Margin"), ax=axes[0,0], set_colors=("skyblue", "#EEDA9F"), alpha = 0.7)
axes[0,0].set_title("Revenue vs Margin")

venn2([set1, set3], set_labels=("Top Revenue", "Top Quantity"), ax=axes[0,1], set_colors=("skyblue", "#fda1cf"), alpha = 0.7)
axes[0,1].set_title("Revenue vs Quantity")

venn2([set2, set3], set_labels=("Top Margin", "Top Quantity"), ax=axes[1,0], set_colors=("#EEDA9F", "#fda1cf"), alpha = 0.7 )
axes[1,0].set_title("Margin vs Quantity")

v3 = venn3([set1,set2, set3], set_labels=("Top Revenue", "Top Margin", "Top Quantity"), ax=axes[1,1], set_colors=("skyblue", "#EEDA9F", "#fda1cf"), alpha = 0.7)
axes[1,1].set_title("All Three Metrics")
label = v3.get_label_by_id('111')
org = label.get_text()
label.set_text(f"{org}\nAll Top")
plt.tight_layout()
plt.show()

In [ ]:
top_items_set = set1 | set2 | set3
top_items = item_inv_agg[item_inv_agg["item"].isin(top_items_set)]

def categorize(item, set1, set2, set3):
    match (item in set1, item in set2, item in set3):
        case (False, False, False):
            return "none"
        case (True, False, False):
            return "Top Revenue only"
        case (False, True, False):
            return "Top Margin only"
        case (False, False, True):
            return "Top Quantity only"
        case (True, True, False):
            return "Top Revenue & Top Margin only"
        case (True, False, True):
            return "Top Revenue & Top Qauntity only"
        case (False, True, True):
            return "Top Margin & Top Quantity only"
        case (True, True, True):
            return "Top Revenue & Top Margin & Top Quantity"

top_items["category"] = top_items["item"].apply(lambda x: categorize(x, set1, set2, set3))
top_items

## Calculate Reorder Point for Top Products

In [ ]:
PROJECT_ID = "clean-pilot-456915-t0"
client = bigquery.Client(project= PROJECT_ID)
df = client.query("""  
SELECT 
    t.id,
    t.tranid,
    t.trandate,
    t.status,
    t.type,
    tl.item,
    i.fullname,
    i.class,
    i.manufacturer,          
    c.category,             
    tl.quantity,
    tl.quantitybackordered,
    tl.netamount,
    tl.rate,
    tl.costestimate,
    tl.inventorylocation,    
    tl.custcolfree_goods_checkbox  
FROM `clean-pilot-456915-t0.NetSuite.transaction` t
JOIN `clean-pilot-456915-t0.NetSuite.transactionline` tl ON t.id = tl.transaction
JOIN `clean-pilot-456915-t0.NetSuite.customer` c ON t.entity = c.id
LEFT JOIN `clean-pilot-456915-t0.NetSuite.item` i ON tl.item = i.id
WHERE t.trandate <= DATE_SUB(CURRENT_DATE(), INTERVAL 1 DAY)
  AND t.type IN ('SalesOrd')   
  AND t.entity NOT IN (604610, 615265)
  AND c.category != 6                      -- exclude wholesale
  AND tl.mainline = FALSE
  AND tl.taxline = FALSE
  AND tl.inventorylocation = 1
  AND tl.itemtype = 'InvtPart'
  AND i.manufacturer != 'D2'
ORDER BY t.trandate DESC
""").to_dataframe()

In [ ]:
so = df.groupby(by = ["tranid", "item"], as_index=False).agg(
    quantity = ("quantity", "sum"), 
    netamount = ("netamount", "sum"),
    costestimate = ("costestimate", "sum"), 
    status = ("status","first"),
    trandate = ("trandate", "first"), 
    type = ("type", "first")
)
# Selected the Analysis timeline
# six_months_ago = (datetime.today() - timedelta(days=182)).date()
# so = so[so["trandate"] >= six_months_ago]

# Adjust the quantitative columns
adj_list = ["quantity", "netamount","costestimate"]
for col in adj_list:
    so[col] = so[col]*-1

# Include 'Paid In Full' only - B
valid_so = so[(so["status"] != "H") & (so["status"] != "C")]

# daily demand per item
daily_demand = valid_so.groupby(by = ["item", "trandate"], as_index = False).agg(daily_demand = ("quantity", "sum"))


In [ ]:

start = datetime(2025,1, 1).strftime('%Y-%m-%d')
end = daily_demand["trandate"].max().strftime('%Y-%m-%d')

daily_demand["trandate"].sort_values
canadian_holidays = [
    "2025-01-01", "2025-02-17", "2025-04-18", "2025-05-19", "2025-07-01",
    "2025-09-01", "2025-09-30", "2025-10-13", "2025-11-11", "2025-12-25",
    "2025-12-26", "2026-01-01", "2026-02-16", "2026-04-03", "2026-05-18",
    "2026-07-01", "2026-09-07", "2026-09-30", "2026-10-12", "2026-11-11",
    "2026-12-25", "2026-12-26"
]
cbd = CustomBusinessDay(holidays=canadian_holidays)
all_business_dates = pd.date_range(start=start, end=end, freq=cbd)
all_items = daily_demand["item"].dropna().drop_duplicates()

bday_index = pd.MultiIndex.from_product([all_business_dates, all_items],
                                         names = ["trandate", "item"])
daily_demand_enriched = (daily_demand.set_index(["trandate", "item"])["daily_demand"]
                            .reindex(bday_index, fill_value=0)
                            .reset_index())

expected_rows = len(all_items) * len(all_business_dates)

print(f"Unique items: {len(all_items):,}")
print(f"Business days: {len(all_business_dates):,}")
print(f"Expected rows: {expected_rows:,}")
print(f"Actual rows: {len(daily_demand_enriched):,}")
print(f"item + trandate is unique: {not daily_demand_enriched.duplicated(['item', 'trandate']).any()}")

In [ ]:
# Filter to only TOP PRODUCTS
daily_demand_enriched[daily_demand_enriched["item"].isin(top_items_set)]

In [ ]:
len(top_items_set)

In [ ]:
start

In [ ]:
end